# Practical Digital Humanities: Winnowing a List of Grants and Fellowships

A colleague is on a committee charged with promoting applications to prestigious grant and fellowship programs. The committee has a list of such fellowships, but they are not organized by discipline. I decided to try using some DH techniques to filter out STEM and other obviously non-humanities grants and fellowships so that our department could more easily identify opportunities for our colleagues. I'm sharing this here because it's a good demonstration of text mining, topic modeling, natural language processing, and Python. 

I'm also sharing it because it could be helpful in the overall effort to increase OU's applications for grants and fellowships.

## Import the necessary packages

The original database is an Excel spreadsheet, because of course it is. That makes it a no-brainer to use the [Pandas](https://pandas.pydata.org/) package to clean, sort, organize, and analyze the data. I will also need Numpy and collections.

For topic modeling, I'll use [BERTopic](https://bertopic.readthedocs.io/en/latest/). It's a bit old-fashioned, but it's useful and straightforward.

For NLP work, I prefer [Spacy](https://spacy.io/) over [NLTK](www.nltk.org). Although Spacy is based on machine learning and more reliant on probabilistic results than NLTK, which is pretty much straight Python, I find Spacy faster and easier to use.

In [72]:
import csv
import pandas as pd
import numpy as np
from collections import Counter

# Import the BERTopic library
from bertopic import BERTopic

# Import the SpaCy library
import spacy
from spacy.tokens import Doc, DocBin

# Load SpaCy's English language model
nlp = spacy.load("en_core_web_lg")

## Read in the original data

Pandas has a built-in function for reading Excel data, but it is important to have the `openpyxl` library in your environment.

In [2]:
df = pd.read_excel("../grants-awards/Faculty_Honorifics_Database.xlsx", sheet_name="Awards Description & Eligibilit", engine="openpyxl")

## Explore the data

I'll do some preliminary analysis to get the lay of the land.

In [3]:
# Examine the structure of the DataFrame
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 19598 entries, 0 to 19597
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   Name                  19598 non-null  str  
 1   GoverningSocietyId    19598 non-null  int64
 2   GoverningSocietyName  19598 non-null  str  
 3   Prestige              1314 non-null   str  
 4   Periodicity           19547 non-null  str  
 5   OpeningMonth          2827 non-null   str  
 6   ClosingMonth          13010 non-null  str  
 7   AnnouncedMonth        13778 non-null  str  
 8   AwardAmount           19598 non-null  int64
 9   AwardURL              19573 non-null  str  
 10  Description           18551 non-null  str  
 11  Eligibility           14366 non-null  str  
dtypes: int64(2), str(10)
memory usage: 1.8 MB


In [4]:
# How many GoverningSocietyName values are there?
society_names = df["GoverningSocietyName"].unique()
print(len(society_names))

3086


I'll create a DocBin for the data processed by Spacy. In Spacy, a "document" is any group of text. In this instance, the names of governing societies will be documents.

In [9]:
# Initialize a DocBin to store the processed documents
society_bin = DocBin(store_user_data=True)

Let's examine the words in the governing societies' names to see if they tell us about the disciplines represented in the database. I'll use Spacy to tokenize the words and remove high-frequency common words like "of" and "the" and punctuation.

In [15]:
word_freq = Counter()
words = []
for society in society_names:
    doc = nlp(society)
    society_bin.add(doc)
    words.extend([token.text for token in doc if not token.is_stop and not token.is_punct])
    word_freq.update([token.text for token in doc if not token.is_stop and not token.is_punct])
word_set = set(words)
# How many unique words are there in the GoverningSocietyName values?
print(f"Number of unique words: {len(word_set)}")

# What are the most common words in the GoverningSocietyName values?
print(f"Most common words: {word_freq.most_common(10)}")

Number of unique words: 3070
Most common words: [('Society', 763), ('Association', 647), ('American', 587), ('Foundation', 443), ('International', 252), ('Research', 234), ('National', 166), ('Institute', 155), ('University', 125), ('Academy', 123)]


As I suspected, the most frequent words are not helpful to me because they don't tell me anything about the disciplines. A topic model of the governing society names might give me more information.

## Topic Modeling

A topic model is a statistical representation of groupings of words in a given text. The "topics" are often pretty abstract, but they can give you a bird's-eye view of the data.

I'll start with a topic model of the governing society names, but I suspect that I'll get a lot of topics about foundations, agencies, societies, etc.

In [16]:
# Make a copy of the dataframe for topic modeling
topic_model_df = df.copy()

In [22]:
# Initialize the BERTopic model
society_topic_model = BERTopic()
# Make sure to use the same vocabulary as the SpaCy model
docs = [doc.text for doc in society_bin.get_docs(nlp.vocab)]
# Fit the BERTopic model to the documents
topics, probs = society_topic_model.fit_transform(docs)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10136.87it/s]


In [23]:
# Inspect the topics generated by the model
society_topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,786,-1_humanities_ultrasound_university_sensing,"[humanities, ultrasound, university, sensing, ...","[Remote Sensing and Photogrammetry Society, Th..."
1,0,86,0_music_guild_musical_composers,"[music, guild, musical, composers, musicologic...","[National Association for Music Education, Soc..."
2,1,75,1_cancer_melanoma_wellness_pancreatic,"[cancer, melanoma, wellness, pancreatic, compr...","[Cancer Research Foundation, V Foundation for ..."
3,2,66,2_library_libraries_librarians_truman,"[library, libraries, librarians, truman, museu...","[Medical Library Association, American Library..."
4,3,54,3_immunology_lesbian_allergy_reproductive,"[immunology, lesbian, allergy, reproductive, g...",[British Society for Allergy and Clinical Immu...
...,...,...,...,...,...
233,232,12,232_collegiate_schools_architecture_aiga,"[collegiate, schools, architecture, aiga, grap...",[Association of Collegiate Schools of Planning...
234,233,12,233_fiction_shock_meteoritical_planetary,"[fiction, shock, meteoritical, planetary, phil...","[World Science Fiction Society, Science Fictio..."
235,234,12,234_aesthetics_standards_ses_masonry,"[aesthetics, standards, ses, masonry, ceramic,...","[British Society of Aesthetics, American Socie..."
236,235,11,235_thomas_jefferson_von_humboldt,"[thomas, jefferson, von, humboldt, golisano, w...","[B. Thomas Golisano Foundation, Thomas Jeffers..."


BERTopic divided the society name data into 78 topics. It looks like the results could be useful in filtering the data, but they're pretty general overall. 

I'll do the same thing for the "Description" column.

In [24]:
description_bin = DocBin(store_user_data=True)
descriptions = topic_model_df['Description'].dropna().unique()
for description in descriptions:
    text = nlp(description)
    doc = Doc(nlp.vocab, words=[token.lemma_ for token in text if not token.is_stop and not token.is_punct])
    description_bin.add(doc)

In [25]:
description_topic_model = BERTopic()
description_docs = [doc.text for doc in description_bin.get_docs(nlp.vocab)]
description_topics, description_probs = description_topic_model.fit_transform(description_docs)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9013.61it/s]


In [26]:
description_topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,6770,-1_research_program_award_support,"[research, program, award, support, project, y...",[ARL Leadership Fellows program executive lead...
1,0,449,0_paper_best_poster_reviewer,"[paper, best, poster, reviewer, good, conferen...",[Best Convention Paper Award give author(s goo...
2,1,263,1_teaching_teacher_educator_educational,"[teaching, teacher, educator, educational, lea...",[recognize individual contribution distinguish...
3,2,240,2_psychology_psychologist_personality_psycholo...,"[psychology, psychologist, personality, psycho...",[award recognize individual distinguished cont...
4,3,201,3_nursing_nurse_palliative_nurses,"[nursing, nurse, palliative, nurses, sigma, hi...",[American Nurses Foundation found educational ...
...,...,...,...,...,...
240,239,11,239_bioengineering_biomedical_bioengineer_engi...,"[bioengineering, biomedical, bioengineer, engi...",[Van C. Mow Medal bestow individual demonstrat...
241,240,10,240_management_strategic_amle_kennesaw,"[management, strategic, amle, kennesaw, ent, p...",[Academy Management Learning Education AMLE Be...
242,241,10,241_earth_space_waldo_smith,"[earth, space, waldo, smith, joanne, agu, simp...",[Waldo E. Smith Award award give biennially nu...
243,242,10,242_occupational_bgc_acoem_hygiene,"[occupational, bgc, acoem, hygiene, hygienist,...",[create 1948 recognize author(s article public...


That looks more promising. I'm going to dig more deeply into this.

In [27]:
description_frame = description_topic_model.get_topic_info()
desc_topics = description_frame['Name'].dropna().unique()
for topic in desc_topics:
    print(topic)

-1_research_program_award_support
0_paper_best_poster_reviewer
1_teaching_teacher_educator_educational
2_psychology_psychologist_personality_psychological
3_nursing_nurse_palliative_nurses
4_service_society_distinguished_member
5_music_composer_opera_musical
6_plant_agricultural_extension_horticultural
7_neuroscience_neurology_neuroradiology_brain
8_mathematical_mathematic_mathematics_mathematician
9_political_politic_good_book
10_dental_dentistry_oral_adea
11_veterinary_animal_veterinarian_avma
12_early_career_scholar_phd
13_psychiatry_mental_psychiatric_adolescent
14_poetry_poet_poem_entry
15_honorary_membership_fellows_elect
16_feminist_gender_woman_women
17_grant_grants_funding_fund
18_cardiovascular_hypertension_council_vascular
19_scholar_visit_seminar_fulbright
20_eye_glaucoma_optometry_vision
21_travel_meeting_aai_expense
22_radiology_acr_spr_radiation
23_journalism_journalist_news_nieman
24_mentor_mentoring_mentorship_mentee
25_space_aerospace_aviation_aiaa
26_optic_optical_ph

As I suspected, the lion's share of the grants and fellowships are STEM-related. But now I can use this information to develop a dictionary of terms I can use to filter out grants and fellowships that are not relevant to my discipline.

I need to do this quickly, so I'm going to make a topic dictionary and ask ChatGPT to categorize the topics for me and return labels that make more sense to a human.

In [28]:
topic_dict = dict(zip(description_frame['Topic'], description_frame['Name']))

Here's the output that I gleaned from ChatGPT's analysis:

In [29]:
excluded_topics = [
    # Medicine, health, clinical sciences
    1,  # cancer / oncology
    5,  # nursing
    6,  # cardiovascular
    8,  # neuroscience / neurology
    14,  # dental / dentistry
    20,  # radiology / radiation
    21,  # physical / sports therapy
    25,  # eye / glaucoma / optometry
    26,  # toxicology
    27,  # psychiatry
    36,  # dermatology
    38,  # nutrition / dietetics
    48,  # aging / geriatrics / gerontology
    51,  # nephrology
    52,  # surgery
    65,  # internal medicine
    66,  # health
    71,  # otolaryngology
    72,  # reproductive / obstetrics / gynecology
    74,  # Alzheimer's / dementia
    76,  # hearing / audiology
    78,  # ultrasound / diagnostic
    86,  # pharmacology / pharmaceutical
    89,  # pharmacy
    90,  # spine / osteopathic
    95,  # immunology / allergy
    98,  # emergency medicine
    101,  # abuse / addiction / drug
    103,  # child psychiatry
    105,  # public health
    107,  # diabetes
    109,  # lung cancer
    112,  # osteopathic / medical
    115,  # endocrinology
    118,  # epilepsy
    123,  # respiratory / pulmonary
    127,  # dairy / nutrition
    129,  # hand surgery
    132,  # sleep medicine
    135,  # occupational therapy
    136,  # transplantation
    138,  # clinical laboratory
    140,  # Parkinson's
    141,  # healthcare simulation
    143,  # anesthesia
    149,  # pediatrics
    150,  # IBD / Crohn's / colitis
    151,  # gastroenterology / nutrition
    152,  # liver / hepatology
    153,  # hemophilia
    164,  # plastic surgery
    168,  # college healthcare
    170,  # nuclear medicine / imaging
    172,  # pain / headache
    173,  # pathology
    179,  # emergency medicine
    186,  # rheumatology / arthritis
    190,  # marriage / family therapy
    191,  # physical therapy
    198,  # gastroenterology
    199,  # lipids
    200,  # pathology
    201,  # neuro-oncology
    205,  # cystic fibrosis
    214,  # cardiothoracic surgery
    218,  # reconstructive surgery
    225,  # multiple sclerosis
    229,  # urology
    236,  # infectious diseases
    238,  # patient care
    # Biological and life sciences
    11,  # plant / agricultural / horticultural
    15,  # veterinary / animal
    33,  # biology / molecular / biochemistry
    77,  # microbiology
    81,  # wildlife / birds
    106,  # food science / safety
    116,  # evolutionary biology
    125,  # forest / forestry
    133,  # physiology
    137,  # ecology
    145,  # entomology
    148,  # genetics
    161,  # soil
    183,  # poultry
    207,  # ornithology
    230,  # fisheries / fishes
    # Chemistry, physics, mathematics, materials
    13,  # mathematics
    32,  # optics / photonics / quantum
    34,  # chemical engineering
    41,  # materials / metals
    45,  # nuclear / reactor
    49,  # physics
    84,  # SIAM / optimization
    87,  # steel / welding
    88,  # combustion / thermal
    114,  # electrochemical / battery
    122,  # ceramics
    139,  # acoustics
    147,  # polymer
    154,  # fluid dynamics / thermodynamics
    163,  # nanotechnology
    181,  # catalysis
    195,  # biomaterials
    197,  # spectroscopy
    209,  # aerosol
    212,  # medicinal chemistry
    222,  # plasma physics
    226,  # crystallography
    227,  # geochemistry
    228,  # chemistry
    232,  # electromagnetics
    # Engineering, computing, technology
    4,  # engineering / manufacturing
    30,  # astronautics / aeronautics
    39,  # IEEE / sensors / antennas
    40,  # IEEE transactions
    42,  # women in engineering
    64,  # structural / construction
    79,  # power / electronics
    159,  # naval / marine / ship
    177,  # signal processing
    192,  # robotics / automation
    204,  # computer vision
    210,  # petroleum engineering
    219,  # bioengineering / biomedical engineering
    235,  # audio / signal processing
    # Earth, atmospheric and physical environmental sciences
    29,  # water / hydrology
    37,  # planetary / astronomy
    47,  # geology
    55,  # energy / solar / renewable
    92,  # atmospheric / climate / meteorology
    184,  # oceanography
    215,  # remote sensing / geoscience
    # Other clearly technical/applied topics
    80,  # statistics / Bayesian
    178,  # mining / minerals / coal
    180,  # photogrammetry / remote sensing
    196,  # strength / conditioning
    208,  # occupational safety / ASSP
    231,  # ergonomics / human factors
]

### Apply the Topic Model to the DataFrame

I'm going to use those topics to filter out the grants and fellowships that aren't relevant to humanities departments.

In [30]:
# Make a mask to select rows where the "Description" column is not null
mask = topic_model_df["Description"].notna()

# Use the description_topic_model to transform the "Description" column into topics and probabilities
topics, probabilities = description_topic_model.transform(topic_model_df.loc[mask, "Description"].tolist())

# Apply the topics and probabilities to the DataFrame
topic_model_df.loc[mask, "Topic"] = topics
topic_model_df.loc[mask, "Probability"] = probabilities

In [31]:
# Cast the "Topic" column to the Int64 data type instead of a float type to avoid NaN values for rows without a description
topic_model_df["Topic"] = topic_model_df["Topic"].astype("Int64")

In [32]:
topic_model_df.head()

,Name,GoverningSocietyId,GoverningSocietyName,Prestige,Periodicity,OpeningMonth,ClosingMonth,AnnouncedMonth,AwardAmount,AwardURL,Description,Eligibility,Topic,Probability
0,#ShareTheMicInCyber Cybersecurity Fellowship,1738,New America,NaN,Annual,January,December,NaN,20000,https://www.newamerica.org/jobs/fellowships/,The #ShareTheMicInCyber Fellowship (#STMIC Fel...,We are looking for individuals who work within...,-1,0.0
1,(See IEEE Computer Society for other awards),3579,The Computer Vision Foundation,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,<NA>,NaN
2,*Abe Fellows (see Social Science Research Coun...,54,American Council of Learned Societies (ACLS),P,Annual,NaN,NaN,NaN,0,http://www.ssrc.org/fellowships/view/abe-fello...,The Abe Fellowship is designed to encourage in...,Past recipients of the Abe Fellowship are inel...,-1,0.0
3,*Abe Fellowship Program,1567,The Japan Foundation,NaN,AnnualRollingDeadline,NaN,September,November,0,https://www.jpf.go.jp/cgp/e/fellow/abe/index.html,The Abe Fellowship Program is a research schol...,This competition is open to citizens of the Un...,-1,0.0
4,*Abraham Pais Prize for History of Physics (se...,311,American Institute of Physics,NaN,Annual,NaN,NaN,NaN,0,NaN,NaN,NaN,<NA>,NaN


I can use Pandas' `isin()` function to remove rows that have been identified as excluded topics.

Note that the tilde (`~`) in the next cell is Pandas-speak for "not".

In [33]:
candidate_df = topic_model_df[~topic_model_df["Topic"].isin(excluded_topics)]

How many rows did that remove? I'll use the `len()` function to find the answer.

In [34]:
print(f"Length of original df: {len(topic_model_df)}")
print(f"Length of candidate df: {len(candidate_df)}")
print(len(topic_model_df) - len(candidate_df), "rows were excluded based on topic filtering.")

Length of original df: 19598
Length of candidate df: 14534
5064 rows were excluded based on topic filtering.


### A pitfall in topic modelling

It's important to be aware that BERTopic will create a catch-all "topic" that includes all the things it couldn't otherwise categorize. Let's find out how many items are in that topic.

In [35]:
topic_model_df["Topic"].value_counts()

Topic
-1     8561
0       942
2       240
3       203
1       166
       ... 
147       3
163       3
228       3
242       2
213       1
Name: count, Length: 244, dtype: Int64

This shows the weakness of relying only on topic modeling: over 8,500 of the items in the database of grants and fellowships are in the "miscellaneous" category. We need a better approach.

Just to show what that topic contains, here is a random sample of twenty descriptions in the "-1" category.

In [36]:
# Show a sample of descriptions that were not assigned to any topic
topic_model_df.loc[topic_model_df["Topic"] == -1, "Description"].sample(20, random_state=42).tolist()

['Recognizes individuals with outstanding contributions in the technical innovation. Encourage step changes associated with cost and speed profiles.',
 'The purposes of this award program are to increase the scientific knowledge base of Women in Cancer Research members and to encourage to pursue careers in cancer research.',
 'Recognizes individuals with outstanding contributions in the pharmaceutical process development.',
 'If you are interested in a stipend for Virtual Symposium, you must apply by September 24th.  There are 10 stipends available to cover virtual registration.',
 "The Ignacio 'Nacho' Tinoco Award of the Biophysical Society honors the scientific contributions, work, and life of an outstanding biophysical chemist, educator, and mentor. Tinoco’s contributions to the spectroscopic, thermodynamic, structural, and single-molecule study of biopolymers consistently deepened our understanding of fundamental biophysical principles, constantly moving this field toward new front

## Natural Language Processing

The topic models did what they were supposed to do: they gave me a bird's-eye view of the entire database of grants and fellowships. I found out that most of the grants and fellowships have to do with STEM topics. That's useful, but I need more detail. I'll use Spacy for more granular detail.

In [37]:
# Make a copy of the dataframe for NLP processing
nlp_df = df.copy()

for org in sorted(nlp_df['GoverningSocietyName'].unique()):
    print(org)

42 Miles Press
92nd Street Young Men's and Young Women's Hebrew Association
A Breath of Hope Lung Foundation
A Public Space
A-T Children's Project
A.P. Giannini Foundation
AABC Foundation
ABSA International (American Biological Safety Association)
ACVO Vision for Animals Foundation
ADEAGies Foundation
ALS Finding a Cure
AO North America
APPA: Leadership in Educational Facilities
ASM International
AUDELCO (Audience Development Committee, Inc.)
AZA Amphibian Taxon Advisory Group
Abdullah bin Hamad Al-Attiyah International Foundation for Energy and Sustainable Development
Abdus Salam International Centre for Theoretical Physics
Academia Europaea
Academia Sinica
Academic Pediatric Association
Academy for Eating Disorders
Academy for Radiology & Biomedical Imaging Research
Academy of American Poets
Academy of Architecture for Health Foundation
Academy of Athens
Academy of Behavioral Medicine Research
Academy of Certified Archivists
Academy of Clinical Laboratory Physicians and Scientists
Ac

I'll make a list of unique words. Spacy makes this easy with its built-in functions and methods. In the next cell, I'll:

1. Use Pandas' `.unique()` method to get the unique values of the "GoverningSocietyName" column.
2. Use Spacy to:

    1. turn that information into a Spacy object
    2. Tokenize the words
    3. Remove high-frequency words (stops)
    4. Remove punctuation
    5. Lemmatize the words so that plurals, possessives, etc. of the same word are treated as one value instead of multiple values

3. Use Python's `sorted()` and `set()` functions to sort the word list and return unique values.


In [39]:
word_list = []
for org in sorted(nlp_df['GoverningSocietyName'].unique()):
    nlporg = nlp(org)
    for token in nlporg:
        if not token.is_stop and not token.is_punct:
            word_list.append(token.lemma_.lower())

unique_words = sorted(set(word_list))
for word in unique_words:
    print(word)

 
+
42
53
90
92nd
a&m
a.
a.p.
aaaai
aaaed
aaas
aabc
aaihs
aajr
aana
aanp
aaom
aaphd
aapl
aasld
aasm
aatf
aauw
abdominal
abdulaziz
abdullah
abdus
absa
abuse
abusers
academia
academic
academies
academy
academyhealth
acbs
access
accountability
accountancy
accountants
accounting
accreditation
accs
ace
acha
achems
achievement
acl
aclm
acls
acm
acne
acoem
acoms
acoustical
acoustics
acta
action
activities
activity
actuarial
actuaries
actuary
acvo
adapted
adara
adces
addiction
addictions
adeagie
adhesion
administration
administrators
admire
admission
admissions
adobe
adolescence
adolescent
adsc
adsorption
adult
advance
advanced
advancement
advances
advancing
advertising
advising
advisor
advisors
advisory
advocacy
aect
aera
aeronautic
aeronautical
aeronautics
aerosol
aerosols
aerospace
aesthetics
affairs
afghanistan
afhvs
afosr
africa
african
agc
age
agents
agesw
agilent
aging
agricultural
agriculture
agronomy
agropolis
aha
ahima
ahme
aiaee
aiche
aid
aided
aids
aiesep
aiga
air
airlie
ajmc
akron

I asked ChatGPT to categorize the word set by discipline.

In [40]:
stem_medical_words = [
    # Engineering and technology
    "aeronautic",
    "aeronautics",
    "aerospace",
    "antenna",
    "astronautics",
    "automation",
    "automotive",
    "biomedical",
    "circuit",
    "computational",
    "computer",
    "computing",
    "construction",
    "cryogenic",
    "cybernetic",
    "cybernetics",
    "electrical",
    "electrochemical",
    "electrochemistry",
    "electromagnetics",
    "electronic",
    "electronics",
    "engineering",
    "engineer",
    "geotechnical",
    "industrial",
    "infrared",
    "instrumentation",
    "machine",
    "machinery",
    "mechanical",
    "mechanics",
    "microwave",
    "robotic",
    "robotics",
    "semiconductor",
    "semiconductors",
    "signal",
    "structural",
    "technical",
    "technological",
    "technologists",
    "transducer",
    "welding",
    # Mathematics, physics, chemistry, materials
    "acoustical",
    "acoustics",
    "actuarial",
    "actuaries",
    "actuary",
    "analytical",
    "bayesian",
    "biophysical",
    "ceramic",
    "chemical",
    "chemistry",
    "chemist",
    "combinatorics",
    "crystallographic",
    "dielectrics",
    "diffraction",
    "electroanalytical",
    "electrostatics",
    "gravitation",
    "gravitational",
    "magnetic",
    "magnetics",
    "materialia",
    "materials",
    "mathematical",
    "mathematics",
    "metallurgical",
    "metallurgy",
    "metals",
    "optical",
    "optics",
    "photonics",
    "physicists",
    "physics",
    "quantum",
    "relativity",
    "rheology",
    "spectrometry",
    "spectroscopy",
    "statistical",
    "statistics",
    "supramolecular",
    "thermal",
    # Biology and life sciences
    "animal",
    "biochemical",
    "biochemistry",
    "biodiversity",
    "biological",
    "biologists",
    "biology",
    "biomaterial",
    "biomechanics",
    "biotechnology",
    "botanical",
    "bovine",
    "bryological",
    "ecological",
    "ecology",
    "entomological",
    "evolutionary",
    "genetical",
    "genetics",
    "genomics",
    "glycobiology",
    "glycoscience",
    "lichenological",
    "mammal",
    "mammalogists",
    "microbial",
    "microbiology",
    "molecular",
    "mycological",
    "nematologists",
    "ornithological",
    "parasitologist",
    "parasitologists",
    "parasitology",
    "phycological",
    "physiological",
    "plant",
    "primatologists",
    "species",
    "zoological",
    # Agriculture, forestry, fisheries
    "agricultural",
    "agriculture",
    "cattle",
    "fisheries",
    "fishes",
    "forest",
    "foresters",
    "horticulture",
    "poultry",
    "soil",
    "soybean",
    "swine",
    "veterinarians",
    "veterinary",
    "wildlife",
    # Earth, atmospheric, ocean and space sciences
    "astronomical",
    "atmospheric",
    "climate",
    "geochemistry",
    "geodesy",
    "geological",
    "geologists",
    "geology",
    "geomorphology",
    "geophysical",
    "geophysicists",
    "geophysics",
    "geoscience",
    "geosciences",
    "geoscientists",
    "hydrogeologists",
    "hydrological",
    "hydrology",
    "limnology",
    "meteorological",
    "mineral",
    "mineralogical",
    "minerals",
    "oceanographic",
    "oceanography",
    "paleontological",
    "planetary",
    "sedimentary",
    "seismological",
    # Medicine — general
    "clinical",
    "clinicians",
    "diagnostic",
    "diagnostics",
    "disease",
    "diseases",
    "healthcare",
    "hospital",
    "medical",
    "medicine",
    "patient",
    "physician",
    "physicians",
    "therapeutics",
    # Medical specialties
    "anesthesia",
    "anesthesiologists",
    "anesthesiology",
    "andrology",
    "cardiology",
    "cardiovascular",
    "dental",
    "dentistry",
    "dermatologic",
    "dermatology",
    "endocrine",
    "endocrinologists",
    "gastroenterological",
    "gastroenterology",
    "gynecologic",
    "gynecologist",
    "gynecologists",
    "gynecology",
    "hematology",
    "nephrology",
    "neurological",
    "neurology",
    "neurosurgery",
    "neurosurgical",
    "obstetric",
    "obstetricians",
    "obstetrics",
    "oncology",
    "ophthalmology",
    "optometry",
    "orthopaedic",
    "orthopedic",
    "osteopathic",
    "osteopathy",
    "otolaryngology",
    "paediatric",
    "pathological",
    "pathologists",
    "pathology",
    "pediatric",
    "pediatrics",
    "pharmaceutical",
    "pharmaceuticals",
    "pharmacists",
    "pharmacology",
    "pharmacy",
    "psychiatric",
    "psychiatrists",
    "psychiatry",
    "pulmonary",
    "radiological",
    "radiologists",
    "radiology",
    "rheumatology",
    "surgeon",
    "surgeons",
    "surgery",
    "surgical",
    "urological",
    "urology",
    # Specific diseases/medical areas
    "addiction",
    "aids",
    "allergy",
    "alzheimer",
    "arthritis",
    "asthma",
    "autism",
    "cancer",
    "diabetes",
    "fibrosis",
    "infection",
    "infectious",
    "leukemia",
    "lymphoma",
    "melanoma",
    "obesity",
    "parkinson",
    "sclerosis",
    "tuberculosis",
    "tumor",
    # Neuroscience / immunology / related biomedical
    "immunogenetics",
    "immunologists",
    "immunology",
    "immunotherapy",
    "neurochemistry",
    "neuroendocrine",
    "neuroendocrinology",
    "neuroimaging",
    "neuromuscular",
    "neurone",
    "neuropathologists",
    "neuroscience",
    "toxicology",
    "vaccines",
    "virology",
    # Nursing, allied health
    "audiology",
    "dietetics",
    "nurse",
    "nurses",
    "nursing",
    "nutrition",
]

In [41]:
medical_words = [
    # General medicine / clinical care
    "clinic",
    "clinical",
    "clinician",
    "diagnostic",
    "diagnostics",
    "disease",
    "disorder",
    "healthcare",
    "hospice",
    "hospital",
    "medical",
    "medicine",
    "patient",
    "physician",
    "practitioner",
    "therapeutics",
    "therapy",
    "treatment",
    # Anatomy / physiology / biomedical
    "anatomist",
    "anatomy",
    "biochemical",
    "biochemistry",
    "biological",
    "biologics",
    "biomaterial",
    "biomechanics",
    "biomedical",
    "biophysical",
    "biotechnology",
    "cell",
    "cytokine",
    "embryo",
    "genetical",
    "genetics",
    "genomics",
    "glycobiology",
    "histocompatibility",
    "metabolic",
    "microbial",
    "microbiology",
    "mitochondrial",
    "molecular",
    "physiological",
    "xenobiotics",
    # Cancer / oncology
    "benign",
    "breast",
    "cancer",
    "fibrolamellar",
    "leukemia",
    "lymphoma",
    "melanoma",
    "oncology",
    "ovarian",
    "prostate",
    "tumor",
    # Cardiology / cardiovascular
    "aneurysm",
    "angiography",
    "cardiology",
    "cardiovascular",
    "echocardiography",
    "heart",
    "hypertension",
    "myocarditis",
    "vascular",
    "vasculitis",
    # Neurology / neuroscience
    "alzheimer",
    "ataxia",
    "brain",
    "cerebral",
    "dementia",
    "neural",
    "neurochemistry",
    "neurocritical",
    "neuroendocrine",
    "neuroendocrinology",
    "neuroimaging",
    "neurological",
    "neurology",
    "neuromuscular",
    "neurone",
    "neuropathologists",
    "neuroscience",
    "neurosurgery",
    "neurosurgical",
    "paralysis",
    "parkinson",
    "sclerosis",
    # Psychiatry / behavioral health
    "addiction",
    "alcoholism",
    "anxiety",
    "depression",
    "mental",
    "psychiatric",
    "psychiatrist",
    "psychiatry",
    "psychopharmacology",
    "psychotherapy",
    "suicide",
    "suicidology",
    # Immunology / infectious disease
    "aids",
    "allergy",
    "asthma",
    "immune",
    "immunogenetics",
    "immunologist",
    "immunology",
    "immunotherapy",
    "infection",
    "infectious",
    "interferon",
    "tuberculosis",
    "vaccine",
    "virology",
    # Hematology
    "blood",
    "haematology",
    "hematology",
    "hematopathology",
    "hemophilia",
    # Endocrinology / metabolic
    "diabetes",
    "endocrine",
    "endocrinologist",
    "obesity",
    # Gastroenterology / digestive / liver
    "alimentary",
    "biliary",
    "digestive",
    "endoscopic",
    "endoscopy",
    "gastroenterological",
    "gastroenterology",
    "gastrointestinal",
    "hepato",
    "hepatology",
    "liver",
    "pancreato",
    # Pulmonary / respiratory
    "lung",
    "pulmonary",
    "respiratory",
    # Nephrology / urology
    "bladder",
    "nephrology",
    "renal",
    "urodynamics",
    "urogenital",
    "urogynecologic",
    "urological",
    "urology",
    # Obstetrics / gynecology / reproductive
    "fetal",
    "gynecologic",
    "gynecologist",
    "gynecology",
    "maternal",
    "neonatal",
    "obstetric",
    "obstetrician",
    "obstetrics",
    "perinatal",
    "perinatology",
    "preeclampsia",
    "reproductive",
    # Pediatrics / adolescent medicine
    "paediatric",
    "pediatric",
    "pediatrics",
    # Surgery
    "arthroscopy",
    "microsurgery",
    "reconstructive",
    "surgeon",
    "surgery",
    "surgical",
    "transplant",
    "transplantation",
    # Orthopedics / musculoskeletal
    "bone",
    "musculoskeletal",
    "orthopaedic",
    "orthopedic",
    "osteopathic",
    "osteopathy",
    "osteoporosis",
    "scoliosis",
    # Dermatology
    "cutaneous",
    "dermatologic",
    "dermatology",
    "eczema",
    "skin",
    # Ophthalmology / vision
    "blindness",
    "cataract",
    "cornea",
    "eye",
    "ophthalmology",
    "opthalmic",
    "optometry",
    "retina",
    # ENT / hearing
    "audiology",
    "auditory",
    "hearing",
    "laryngological",
    "otolaryngology",
    "otological",
    "rhinologic",
    "rhinological",
    # Dentistry / oral health
    "dental",
    "dentistry",
    "endodontists",
    "maxillofacial",
    "periodontology",
    "prosthodontics",
    # Radiology / medical imaging
    "imaging",
    "radiological",
    "radiologist",
    "radiology",
    "radiosurgery",
    "stereotactic",
    "ultrasound",
    # Pathology / laboratory medicine
    "cytopathology",
    "histotechnology",
    "laboratory",
    "pathological",
    "pathologist",
    "pathology",
    # Pharmacology / pharmacy
    "drug",
    "pharmaceutical",
    "pharmacist",
    "pharmacoepidemiology",
    "pharmacognosy",
    "pharmacology",
    "pharmacometrics",
    "pharmacy",
    # Epidemiology / public-health-specific
    "epidemiologic",
    "epidemiological",
    "epidemiologist",
    "epidemiology",
    # Toxicology
    "toxicology",
    # Nutrition / dietetics
    "dietetics",
    "nutrition",
    # Specific diseases / syndromes appearing in the list
    "angelman",
    "blepharospasm",
    "cystic",
    "dyslexia",
    "dystrophy",
    "eosinophilic",
    "fibrosis",
    "frontotemporal",
    "histiocytosis",
    "hyperoxaluria",
    "ichthyosis",
    "lupus",
    "myasthenia",
    "myositis",
    "oxalosis",
    "palsy",
    "prader",
    "rett",
    "tuberous",
    "tourette",
    "vulvodynia",
]

I'll narrow that down further by lemmatizing the values in those word lists.

In [42]:
stem_lemmas = [nlp(word)[0].lemma_.lower() for word in stem_medical_words]
stem_lemma_set = sorted(set(stem_lemmas))
medical_lemmas = [nlp(word)[0].lemma_.lower() for word in medical_words]
medical_lemma_set = sorted(set(medical_lemmas))

Now I'll try to identify which governing societies have STEM and/or medical terms in their name.

In [44]:
# Initialize a dictionary to store the STEM organizations and their matching stemmed lemmas
stem_orgs = {}

# Loop through the unique GoverningSocietyName values in the DataFrame
for org in nlp_df["GoverningSocietyName"].dropna().unique():
    # Get the lemmas for the organization name, excluding stop words and punctuation
    lemmas = {
        # Use casefold() to normalize the lemmas for comparison
        token.lemma_.casefold()
        for token in nlp(org)
        if not token.is_stop and not token.is_punct
    }

    matches = lemmas & set(stem_lemma_set)

    if matches:
        stem_orgs[org] = sorted(matches)

# Initialize a dictionary to store the medical organizations and their matching medical lemmas
medical_orgs = {}

# Loop through the unique GoverningSocietyName values in the DataFrame
for org in nlp_df["GoverningSocietyName"].dropna().unique():
    lemmas = {
        # Use casefold() to normalize the lemmas for comparison
        token.lemma_.casefold()
        for token in nlp(org)
        if not token.is_stop and not token.is_punct
    }

    matches = lemmas & set(medical_lemma_set)

    if matches:
        medical_orgs[org] = sorted(matches)

I'm going to make a combined list of STEM and medical organizations.

In [45]:
# Initialize a list to store the STEM organization names
stem_org_list = []
# Loop through the stem_orgs dictionary and append the keys (organization names) to the stem_org_list
for k,v in stem_orgs.items():
    stem_org_list.append(k)

# Initialize a list to store the medical organization names
medical_org_list = []
# Loop through the medical_orgs dictionary and append the keys (organization names) to the medical_org_list
for k,v in medical_orgs.items():
    medical_org_list.append(k)

# Create a combined list of unique organization names from both STEM and medical organizations
combined_orgs = sorted(set(stem_org_list) | set(medical_org_list))

In [46]:
len(combined_orgs)

933

I'll make a new DataFrame that contains only the rows where the "GoverningSocietyName" is not in the combined list of STEM and medical organizations.

In [47]:
candidate_2 = nlp_df[~nlp_df["GoverningSocietyName"].isin(combined_orgs)]

In [49]:
len(nlp_df), len(candidate_2), len(nlp_df) - len(candidate_2)

(19598, 12517, 7081)

That removed 7,081 records. Not bad, but I can do more. I'm going to use my lists of STEM and medical words from earlier to find out how many words in a row's "Description" column are STEM or medical in nature. The idea is that I can eventually set a target number of matches that will exclude a row as having a high probability of being STEM or medical in nature.

In [50]:
# Combine the word lists I made earlier into a single set for exclusion matching 
excluded_vocab = set(stem_medical_words) | set(medical_words)

# Define a function to get the exclusion matches for a given text
def get_exclusion_matches(text):
    """
    Get the exclusion matches for a given text.
    This function takes a string of text as input, processes it using SpaCy's NLP model, and returns a sorted list of unique lemmas that match the excluded vocabulary.

    Arguments:
        text (str): The input text to process.
    
    Returns:
        list: A sorted list of unique lemmas that match the excluded vocabulary. If the input text is NaN, an empty list is returned.
    """
    # Check if the input text is NaN (not a number) using pandas' isna function. If it is, return an empty list.
    if pd.isna(text):
        return []

    # Process the input text using SpaCy's NLP model to create a Doc object
    doc = nlp(text)

    # Extract the lemmas from the Doc object
    lemmas = {
        token.lemma_.casefold()
        for token in doc
        if token.is_alpha and not token.is_stop and not token.is_punct
    }

    return sorted(lemmas & excluded_vocab)

# Apply the get_exclusion_matches function to the "Description" column of candidate_2 DataFrame 
# and create a new column "description_exclusion_matches" to store the results
candidate_2["description_exclusion_matches"] = candidate_2["Description"].apply(
    get_exclusion_matches
)

I'll calculate the number of exclusion matches for each description and store it in a new column `n_exclusion_matches`.

In [51]:
candidate_2["n_exclusion_matches"] = candidate_2[
    "description_exclusion_matches"
].str.len()

Now I'll experiment with the number of exclusion matches that will give me reasonable results. This is a subjective move, but that's why there's a human in the loop, right?

I'll try greater than or equal to two exclusion matches.

In [56]:
candidate_2.loc[
    candidate_2["n_exclusion_matches"] >= 2,
    ["Description", "description_exclusion_matches"],
]

,Description,description_exclusion_matches
43,The annual Julius Shiskin Award for Economic S...,"[statistical, statistics]"
45,Foundation of the American Society of Neurorad...,"[radiologists, radiology]"
66,The A. E. Conrady Award is given annually in r...,"[construction, instrumentation, optical]"
89,The A. Richard Newton Educator ABIE Award reco...,"[automation, circuit, computing, electronic, e..."
115,Formerly ôAward for International Scientific C...,"[engineer, engineering]"
...,...,...
19532,The North American Spine Society (NASS) is a p...,"[clinical, disorder, healthcare, medical, neur..."
19535,The YIP is open to United States citizens and/...,"[engineer, engineering]"
19538,The National Kidney Foundation recognizes the ...,"[clinical, disease, patient, treatment]"
19540,The Japan Institute of Metals and Materials (J...,"[industrial, materials, metals]"


That narrows it down to 1,319 rows. 

Now I'll try greater than or equal to three exclusion matches.

In [57]:
candidate_2.loc[
    candidate_2["n_exclusion_matches"] >= 3,
    ["Description", "description_exclusion_matches"],
]


,Description,description_exclusion_matches
66,The A. E. Conrady Award is given annually in r...,"[construction, instrumentation, optical]"
89,The A. Richard Newton Educator ABIE Award reco...,"[automation, circuit, computing, electronic, e..."
255,This award recognizes an individual who has ma...,"[disease, immune, therapy]"
291,The Scholarship Award is to be used in an acad...,"[clinical, diagnostic, disease, epidemiology, ..."
293,The American Association of Obstetricians and ...,"[clinical, gynecologists, gynecology, obstetri..."
...,...,...
19404,The World Laureates Association Prize (WLA Pri...,"[computer, mathematics, medicine]"
19532,The North American Spine Society (NASS) is a p...,"[clinical, disorder, healthcare, medical, neur..."
19538,The National Kidney Foundation recognizes the ...,"[clinical, disease, patient, treatment]"
19540,The Japan Institute of Metals and Materials (J...,"[industrial, materials, metals]"


That's more conservative, which is good. I don't want to remove too many rows automatically because there might be some false matches in the version with 1,319 exclusion matches.

It would be good to approach this from the other end, too. I'll look for positive matches to terms related to the humanities. I'm just going to brainstorm a list of humanities terms, then I'll use NLP to narrow it down.

In [59]:
# Make a list of humanities-related terms
humanities_terms = [
    "history",
    "literature",
    "philosophy",
    "classics",
    "archaeology",
    "art",
    "language",
    "ethics",
    "religion",
    "religious",
    "theology",
    "anthropology",
    "english",
    "poetry",
    "poetics",
    "linguistics",
    "numismatics",
    "philology",
    "classical",
    "humanities",
    "cultural",
    "historical",
    "philosophical",
    "literary",
    "mythology",
    "myth",
    "folklore",
    "folkloric",
    "archive",
    "archival",
    "library",
    "libraries",
    "museum",
    "museums",
    "poetry",
    "poetic",
    "drama",
    "theater",
    "theatre",
    "comedy",
    "tragedy"
]

In [60]:
humanities_lemmas = [nlp(term)[0].lemma_.lower() for term in humanities_terms]
humanities_lemma_set = set(humanities_lemmas)

In [62]:
# Define a function to get the positive matches for a given text
def get_positive_matches(text):
    """
    Get the positive matches for a given text.
    This function takes a string of text as input, processes it using SpaCy's NLP model, and returns a sorted list of unique lemmas that match the humanities-related vocabulary.
    
    Arguments:
        text (str): The input text to process.
        
    Returns:
        list: A sorted list of unique lemmas that match the humanities-related vocabulary.
    """
    if pd.isna(text):
        return []

    doc = nlp(text)

    lemmas = {
        token.lemma_.casefold()
        for token in doc
        if token.is_alpha and not token.is_stop and not token.is_punct
    }

    return sorted(lemmas & humanities_lemma_set)



candidate_2["description_positive_matches"] = candidate_2["Description"].apply(
    get_positive_matches
)

I'll make an `n_positive_matches` column that shows how many humanities terms are in each row's "Description" column.

In [63]:
candidate_2["n_positive_matches"] = candidate_2[
    "description_positive_matches"
].str.len()

In [65]:
candidate_2

,Name,GoverningSocietyId,GoverningSocietyName,Prestige,Periodicity,OpeningMonth,ClosingMonth,AnnouncedMonth,AwardAmount,AwardURL,Description,Eligibility,description_exclusion_matches,n_exclusion_matches,description_positive_matches,n_positive_matches
0,#ShareTheMicInCyber Cybersecurity Fellowship,1738,New America,NaN,Annual,January,December,NaN,20000,https://www.newamerica.org/jobs/fellowships/,The #ShareTheMicInCyber Fellowship (#STMIC Fel...,We are looking for individuals who work within...,[technical],1,[],0
2,*Abe Fellows (see Social Science Research Coun...,54,American Council of Learned Societies (ACLS),P,Annual,NaN,NaN,NaN,0,http://www.ssrc.org/fellowships/view/abe-fello...,The Abe Fellowship is designed to encourage in...,Past recipients of the Abe Fellowship are inel...,[],0,[],0
3,*Abe Fellowship Program,1567,The Japan Foundation,NaN,AnnualRollingDeadline,NaN,September,November,0,https://www.jpf.go.jp/cgp/e/fellow/abe/index.html,The Abe Fellowship Program is a research schol...,This competition is open to citizens of the Un...,[],0,[],0
7,*ACSA/AIA Housing Design Education Award (see ...,70,American Institute of Architects,NaN,Annual,NaN,NaN,NaN,0,na,NaN,NaN,[],0,[],0
10,*CAA/AIC Award for Distinction in Scholarship ...,1289,American Institute for Conservation/Foundation...,NaN,Annual,NaN,NaN,NaN,0,na,NaN,NaN,[],0,[],0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19593,Zoetis Physiology Award,57,American Dairy Science Association,P,Annual,October,January,July,0,http://www.adsa.org/Membership/ADSAAwards.aspx,To recognize outstanding research in dairy cat...,1. Research work published in refereed journal...,[cattle],1,[],0
19594,Zora Neale Hurston Prize,65,American Folklore Society,NaN,AnnualRollingDeadline,NaN,August,November,250,https://americanfolkloresociety.org/our-work/p...,This prize is named for the pioneering folklor...,Works submitted for prize consideration do not...,[eye],1,[folklore],1
19595,Zounon Victorine Award,352,International Studies Association,NaN,Annual,NaN,October,March,1000,https://www.isanet.org/Programs/Awards/Zounon-...,The Zounon Victorine Award is an annual recogn...,Applicants must be active members of the Inter...,[],0,[],0
19596,Zukunftskolleg Lecture,2908,Universität Konstanz,NaN,Sporadic,NaN,NaN,NaN,0,https://www.uni-konstanz.de/zukunftskolleg/eve...,"On a regular basis, the Zukunftskolleg Lecture...",NaN,[],0,[],0


Instead of trying different values for `n` as I did above, I can use Pandas' `value_counts()` method to count the number of matches.

In [67]:
candidate_2["n_positive_matches"].value_counts().sort_index()

n_positive_matches
0     9991
1     1700
2      513
3      184
4       68
5       25
6       17
7        4
8        7
9        7
10       1
Name: count, dtype: int64

I'll do the same for the `n_exclusion_matches`.

In [68]:
candidate_2["n_exclusion_matches"].value_counts().sort_index()

n_exclusion_matches
0     9452
1     1746
2      681
3      279
4      158
5       70
6       44
7       27
8       23
9       10
10      11
11       4
12       3
13       2
14       2
15       1
16       1
17       1
18       1
20       1
Name: count, dtype: int64

I'm going to make a final DataFrame of rows where:

1. The `n_exclusion_matches` is less than or equal to 2, to be conservative
2. The `n_positive_matches` is greater than or equal to 2, to be inclusive of humanities terms

In [77]:
final = candidate_2[(candidate_2["n_exclusion_matches"] <= 2) & (candidate_2["n_positive_matches"] >= 2)].copy()

print(f"Length of final DataFrame: {len(final)}")
print(f"NLP removed {len(candidate_2) - len(final)} rows from candidate_2 based on exclusion and positive match filtering.")
print(f"Percentage of candidate_2 removed: {(len(candidate_2) - len(final)) / len(candidate_2) * 100:.2f}%")

Length of final DataFrame: 812
NLP removed 11705 rows from candidate_2 based on exclusion and positive match filtering.
Percentage of candidate_2 removed: 93.51%


That's not a bad result: 812 records out of the original 19,597. That's a reduction of about 94%! It's still a lot to comb through for specific fields in the humanities, but it's more manageable and focused. 

## Write the data to CSV and Excel

I'll write the data to both CSV and Excel, because, you know, people like Excel for some reason.

In [73]:
final.to_csv(
    "../grants-awards/filtered_candidate_awards.csv",
    index=False,
    encoding="utf-8",
    quoting=csv.QUOTE_ALL,
)
final.to_excel(
    "../grants-awards/filtered_candidate_awards.xlsx", index=False, engine="openpyxl"
)